# Lesson 03 - Агентские шаблоны проектирования

В этом уроке мы изучаем три фундаментальных шаблона проектирования для создания эффективных AI-агентов:

1. **Чёткие инструкции для агента** — создание точных, определяющих роль подсказок, которые направляют поведение агента  
2. **Структурированный вывод с помощью моделей Pydantic** — обеспечение возврата агентами предсказуемых, проверенных данных  
3. **Агенты с одной ответственностью** — проектирование сфокусированных агентов, которые выполняют по одной задаче хорошо  

Мы применим каждый шаблон к сценарию **рекомендателя туристических направлений**, постепенно создавая систему, которая сможет предлагать направления, проверять их доступность и заниматься логистикой.


## Настройка


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Шаблон 1: Четкие инструкции для агента

Самый эффективный шаблон также является самым простым: написание четких, подробных инструкций для вашего агента.

Хорошие инструкции определяют:
- **Кто** такой агент (персона и тон)
- **Что** он должен делать (пошаговые обязанности)
- **Как** он должен себя вести (ограничения и стиль)

Ниже мы создаем агента-консьержа для путешествий с явными инструкциями, которые формируют каждый его ответ.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Шаблон 2: Структурированный вывод с моделями Pydantic

Свободный текст полезен для разговора, но последующим системам нужны структурированные данные.
Сочетая **модели Pydantic** с **функцией инструмента**, мы можем:

- Определить точную схему для вывода агента
- Автоматически проверять ответы
- Надежно интегрировать результаты агента в логику приложения

Мы также представляем инструмент, который возвращает сведения о назначении, чтобы агент основывал свои рекомендации на реальных данных.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Паттерн 3: Агент с одной ответственностью

Сложные задачи выигрывают от распределения работы между несколькими специализированными агентами, каждый из которых отвечает за свою единственную задачу:

- **Эксперт по направлениям**, который знает о местах и доступности
- **Планировщик логистики**, который занимается рейсами, отелями и маршрутами

Это отражает принцип инженерии программного обеспечения *разделения ответственности* — каждый агент легче тестировать, поддерживать и улучшать независимо.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Резюме

В этом уроке мы применили три паттерна агентного дизайна к сценарию рекомендателя путешествий:

| Паттерн | Ключевая идея | Преимущество |
|---|---|---|
| **Четкие инструкции** | Заранее определить персону, обязанности и ограничения | Последовательное поведение агента в рамках бренда |
| **Структурированный вывод** | Использовать модели Pydantic в качестве формата ответа | Проверяемые, машиночитаемые результаты |
| **Принцип единственной ответственности** | Давать каждому агенту одну узконаправленную задачу | Легче тестировать, поддерживать и компоновать |

Эти паттерны естественно сочетаются — вы можете комбинировать четкие инструкции со структурированным выводом внутри агента с одной ответственностью, чтобы создавать надежные, готовые к производству системы.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:  
Этот документ был переведен с помощью сервиса автоматического перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия обеспечить точность, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Настоящий документ на оригинальном языке следует считать авторитетным источником. Для важной информации рекомендуется профессиональный перевод с участием человека. Мы не несем ответственности за любые недоразумения или неверные толкования, возникшие в результате использования данного перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
